# 01 - Data Collection

**Author:** Juan Ruelas  
**Date:** 2026-05

This notebook downloads the two raw inputs we need for the project:

1. **Germany hourly electricity load** from the Open Power System Data (OPSD) time-series package, which mirrors ENTSO-E Transparency Platform values.
2. **Berlin hourly historical weather** from the free Open-Meteo Archive API.

Both sources are public and require no API key. We keep the raw CSVs untouched in `../data/raw/` so the downstream notebooks have a reproducible starting point even if upstream APIs change.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

raw_dir = Path('..') / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# berlin coords - weather pull
berlin = {'latitude': 52.52, 'longitude': 13.41}
start_date = '2015-01-01'
end_date   = '2019-12-31'

## 1. Energy data - Germany hourly load (ENTSO-E via OPSD)

OPSD publishes a tidy, hourly CSV with one column per country. We only need `utc_timestamp` and `DE_load_actual_entsoe_transparency`. The file is roughly 300 MB raw, so we use `usecols` to keep memory low and skip everything we will not consume.

We then trim to the project window (2015-2019) - the early ENTSO-E years are less complete and 2020 onward is influenced by the COVID demand shock, which would confound the weather signal we want to model.

In [ ]:
opsd_url = 'https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv'

load = pd.read_csv(
    opsd_url,
    usecols=['utc_timestamp', 'DE_load_actual_entsoe_transparency'],
    parse_dates=['utc_timestamp'],
)
load = load.rename(columns={
    'utc_timestamp': 'timestamp',
    'DE_load_actual_entsoe_transparency': 'load_mw',
})
load = load.dropna(subset=['load_mw']).reset_index(drop=True)
load = load[(load['timestamp'] >= start_date) & (load['timestamp'] <= end_date + ' 23:00:00')]
load['timestamp'] = pd.to_datetime(load['timestamp'], utc=True).dt.tz_convert(None)
load = load.sort_values('timestamp').reset_index(drop=True)

print(f'shape: {load.shape}')
print('range:', load['timestamp'].min(), '->', load['timestamp'].max())
load.head()

In [ ]:
load_path = raw_dir / 'germany_load.csv'
load.to_csv(load_path, index=False)
print('saved to', load_path)

## 2. Weather data - Berlin hourly (Open-Meteo Archive)

Berlin is a defensible proxy for German national demand because it sits in the country's population-weighted geographic centroid. We request four hourly variables that are known load drivers:

- **temperature_2m** - heating and cooling demand.
- **relative_humidity_2m** - perceived temperature, AC load in summer.
- **wind_speed_10m** - building heat loss; loosely correlates with wind generation, which can shift residual load.
- **direct_radiation** - daylight proxy and rooftop-PV self-consumption.

The Open-Meteo archive returns ERA5 reanalysis values, so we have a clean, homogeneous, gap-free record.

In [ ]:
openmeteo_url = 'https://archive-api.open-meteo.com/v1/archive'

params = {
    **berlin,
    'start_date': start_date,
    'end_date':   end_date,
    'hourly': 'temperature_2m,relative_humidity_2m,wind_speed_10m,direct_radiation',
    'timezone': 'UTC',
}

resp = requests.get(openmeteo_url, params=params, timeout=120)
resp.raise_for_status()
payload = resp.json()

weather = pd.DataFrame(payload['hourly'])
weather = weather.rename(columns={'time': 'timestamp'})
weather['timestamp'] = pd.to_datetime(weather['timestamp'])
weather = weather.sort_values('timestamp').reset_index(drop=True)

print('shape:', weather.shape)
print('nulls:\n', weather.isna().sum())
weather.head()

In [ ]:
weather_path = raw_dir / 'berlin_weather.csv'
weather.to_csv(weather_path, index=False)
print('saved to', weather_path)

## 3. Summary

Two raw CSVs are now staged in `../data/raw/`:

| File | Rows | Columns | Source |
|---|---|---|---|
| `germany_load.csv` | ~43.8k hourly | timestamp, load_mw | OPSD / ENTSO-E TP |
| `berlin_weather.csv` | ~43.8k hourly | timestamp + 4 weather features | Open-Meteo ERA5 archive |

Both share a UTC hourly index and a 5-year span (2015-2019), which gives us a clean join key in notebook 02.